# 03B — Classificação do comprador (setor público) — versão out/2026

**Regra.** Entram na base pública apenas compras de órgãos e entidades da administração
pública direta e indireta (Brasil, 1967 — Decreto-Lei nº 200, art. 4º), identificados pela
**natureza jurídica do CNPJ do comprador** no cadastro da Receita Federal de dezembro de 2023
(resultado da rotina 03C):
- grupo 1 da Tabela de Natureza Jurídica (Administração Pública), e
- 201-1 Empresa Pública e 203-8 Sociedade de Economia Mista (administração indireta).

**Saem:** registros sem CNPJ do comprador (não é possível identificar a natureza jurídica),
CNPJs não encontrados na Receita e compradores de qualquer outra natureza jurídica
(entidades empresariais privadas, associações, fundações privadas, serviços sociais
autônomos etc.).

**Substitui** a versão anterior, que classificava pelo campo "esfera" do BPS (existente só até
2016) e propagava por CNPJ para 2017–2023.

**Entrada:** `Base Harmonizacao Campos` (rotina 03) e `Base Classificacao Receita` (rotina 03C)
**Saída:** `Base Harmonizacao Publica` (+ `_excluidos`, fluxo e transição)

In [1]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
# Ajuste PASTA_DADOS para o local onde estao os arquivos.
# Padrao: subpasta "dados" ao lado do notebook. No Google Colab, aponte para
# a pasta do seu Drive apos monta-lo.
from google.colab import drive
drive.mount('/content/drive')

import os
PASTA_DADOS = os.environ.get('BPS_DADOS', '/content/drive/MyDrive/TCC/Registro de Compras v2/Base v2')
# ─────────────────────────────────────────────────────────────────────────

Mounted at /content/drive


In [2]:
!pip install polars==1.35.2 --quiet

In [3]:
from pathlib import Path
import polars as pl
pl.Config.set_tbl_rows(-1); pl.Config.set_fmt_str_lengths(60); pl.Config.set_tbl_width_chars(200)

BASE          = Path(PASTA_DADOS)
PASTA_HARM    = BASE / 'Base Harmonizacao Campos'
PASTA_RECEITA = BASE / 'Base Classificacao Receita'
PASTA_SAIDA   = BASE / 'Base Harmonizacao Publica'
PASTA_EXCL    = PASTA_SAIDA / '_excluidos'
ANOS = range(2009, 2024)

NATUREZAS_PUBLICAS_FORA_DO_GRUPO_1 = {'2011': 'Empresa Publica', '2038': 'Sociedade de Economia Mista'}

for p in [PASTA_HARM, PASTA_RECEITA]:
    print(f'{"OK " if p.exists() else "FALTANDO"}  {p}')
assert PASTA_HARM.exists() and PASTA_RECEITA.exists(), 'Confira PASTA_DADOS e se o Drive está montado.'
assert (PASTA_RECEITA / 'compradores_receita.parquet').exists(), 'Rode a rotina 03C antes.'

OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Harmonizacao Campos
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Classificacao Receita


## 1. Classificação de cada CNPJ comprador

In [4]:
receita = pl.read_parquet(PASTA_RECEITA / 'compradores_receita.parquet').select(
    'cnpj', 'razao_social', 'encontrado_receita', 'natureza_cod', 'natureza_desc', 'grupo_natureza', 'ente_federativo')
receita = receita.with_columns(
    pl.when(~pl.col('encontrado_receita')).then(pl.lit('nao_encontrado_receita'))
      .when((pl.col('grupo_natureza') == 'Administracao Publica')
            | pl.col('natureza_cod').is_in(list(NATUREZAS_PUBLICAS_FORA_DO_GRUPO_1)))
      .then(pl.lit('publico'))
      .otherwise(pl.lit('natureza_privada')).alias('classe_comprador'))
print(receita.group_by('classe_comprador').len().sort('classe_comprador'))

shape: (2, 2)
┌──────────────────┬──────┐
│ classe_comprador ┆ len  │
│ ---              ┆ ---  │
│ str              ┆ u32  │
╞══════════════════╪══════╡
│ natureza_privada ┆ 54   │
│ publico          ┆ 2027 │
└──────────────────┴──────┘


## 2. Situação anterior (para a tabela de transição)
Lê a base pública gerada pela versão anterior, se existir, ANTES de sobrescrevê-la.

In [5]:
CNPJ = (pl.col('cnpj_instituicao').cast(pl.Utf8).str.replace_all(r'\D', ''))
def cnpj14(df):
    return df.with_columns(pl.when(CNPJ.str.len_chars() > 0).then(CNPJ.str.pad_start(14, '0'))
                             .otherwise(None).alias('cnpj'))

antigos = [PASTA_SAIDA / f'BPS_{a}.parquet' for a in ANOS]
if all(a.exists() for a in antigos):
    ant = pl.concat([cnpj14(pl.read_parquet(a, columns=['cnpj_instituicao'])) for a in antigos])
    ant_reg = ant.height
    ant_cnpjs = set(ant['cnpj'].drop_nulls().to_list())
    print(f'Versão anterior: {ant_reg:,} registros | {len(ant_cnpjs):,} CNPJs')
else:
    ant_reg, ant_cnpjs = None, None
    print('Versão anterior não encontrada — a tabela de transição não será gerada.')

Versão anterior não encontrada — a tabela de transição não será gerada.


## 3. Aplicação da regra, ano a ano

In [6]:
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
PASTA_EXCL.mkdir(parents=True, exist_ok=True)
# remove saídas da versão anterior que deixaram de existir
for velho in ['casos_revisao_manual.csv']:
    if (PASTA_SAIDA / velho).exists():
        (PASTA_SAIDA / velho).unlink(); print('removido (versão anterior):', velho)
for f in PASTA_EXCL.glob('BPS_*_excluidos.parquet'):
    f.unlink()

fluxo, usados = [], []
for ano in ANOS:
    df = cnpj14(pl.read_parquet(PASTA_HARM / f'BPS_{ano}.parquet'))
    df = df.join(receita, on='cnpj', how='left').with_columns(
        pl.when(pl.col('cnpj').is_null()).then(pl.lit('sem_cnpj'))
          .when(pl.col('classe_comprador').is_null()).then(pl.lit('nao_encontrado_receita'))
          .otherwise(pl.col('classe_comprador')).alias('classe_comprador'))
    pub  = df.filter(pl.col('classe_comprador') == 'publico')
    excl = df.filter(pl.col('classe_comprador') != 'publico')
    pub.drop('cnpj', 'classe_comprador', 'encontrado_receita', 'grupo_natureza', 'razao_social').write_parquet(PASTA_SAIDA / f'BPS_{ano}.parquet')
    if excl.height:
        excl.drop('cnpj', 'encontrado_receita', 'razao_social').rename({'classe_comprador': 'motivo_exclusao'}) \
            .write_parquet(PASTA_EXCL / f'BPS_{ano}_excluidos.parquet')
    fluxo.append(dict(ano=ano, total=df.height, publico=pub.height,
                      sem_cnpj=(df['classe_comprador'] == 'sem_cnpj').sum(),
                      natureza_privada=(df['classe_comprador'] == 'natureza_privada').sum(),
                      nao_encontrado_receita=(df['classe_comprador'] == 'nao_encontrado_receita').sum()))
    usados.append(df.select('cnpj', 'classe_comprador'))

fluxo = pl.DataFrame(fluxo)
tot = fluxo.select(pl.lit('TOTAL').alias('ano'), pl.exclude('ano').sum())
fluxo_out = pl.concat([fluxo.with_columns(pl.col('ano').cast(pl.Utf8)), tot])
fluxo_out.write_csv(PASTA_SAIDA / 'fluxo_classificacao_comprador.csv')
print(fluxo_out)

shape: (16, 6)
┌───────┬────────┬─────────┬──────────┬──────────────────┬────────────────────────┐
│ ano   ┆ total  ┆ publico ┆ sem_cnpj ┆ natureza_privada ┆ nao_encontrado_receita │
│ ---   ┆ ---    ┆ ---     ┆ ---      ┆ ---              ┆ ---                    │
│ str   ┆ i64    ┆ i64     ┆ i64      ┆ i64              ┆ i64                    │
╞═══════╪════════╪═════════╪══════════╪══════════════════╪════════════════════════╡
│ 2009  ┆ 13463  ┆ 5111    ┆ 27       ┆ 8325             ┆ 0                      │
│ 2010  ┆ 19066  ┆ 10072   ┆ 38       ┆ 8956             ┆ 0                      │
│ 2011  ┆ 13611  ┆ 8306    ┆ 99       ┆ 5206             ┆ 0                      │
│ 2012  ┆ 4875   ┆ 3217    ┆ 0        ┆ 1658             ┆ 0                      │
│ 2013  ┆ 3219   ┆ 2169    ┆ 9        ┆ 1041             ┆ 0                      │
│ 2014  ┆ 20973  ┆ 19424   ┆ 277      ┆ 1272             ┆ 0                      │
│ 2015  ┆ 99749  ┆ 98009   ┆ 100      ┆ 1640             ┆ 0 

## 4. Compradores excluídos por natureza jurídica

In [7]:
todos = pl.concat(usados)
excl_nat = (todos.filter(pl.col('classe_comprador') == 'natureza_privada')
    .group_by('cnpj').len().join(receita, on='cnpj', how='left')
    .group_by('natureza_cod', 'natureza_desc').agg(pl.len().alias('cnpjs'), pl.col('len').sum().alias('registros'))
    .sort('registros', descending=True))
print(excl_nat)
excl_nat.write_csv(PASTA_SAIDA / 'excluidos_por_natureza.csv')

shape: (4, 4)
┌──────────────┬───────────────────────────────┬───────┬───────────┐
│ natureza_cod ┆ natureza_desc                 ┆ cnpjs ┆ registros │
│ ---          ┆ ---                           ┆ ---   ┆ ---       │
│ str          ┆ str                           ┆ u32   ┆ u32       │
╞══════════════╪═══════════════════════════════╪═══════╪═══════════╡
│ 3999         ┆ Associação Privada            ┆ 46    ┆ 27046     │
│ 3069         ┆ Fundação Privada              ┆ 6     ┆ 8816      │
│ 3077         ┆ Serviço Social Autônomo       ┆ 1     ┆ 179       │
│ 2062         ┆ Sociedade Empresária Limitada ┆ 1     ┆ 2         │
└──────────────┴───────────────────────────────┴───────┴───────────┘


## 5. Transição em relação à versão anterior
Quais CNPJs entraram e quais saíram da base pública com a nova regra.

In [8]:
if ant_cnpjs is not None:
    novos = set(todos.filter(pl.col('classe_comprador') == 'publico')['cnpj'].to_list())
    reg = todos.filter(pl.col('cnpj').is_not_null()).group_by('cnpj').len()
    trans = (receita.join(reg, on='cnpj', how='left')
        .with_columns(pl.col('cnpj').is_in(list(ant_cnpjs)).alias('antes'), pl.col('cnpj').is_in(list(novos)).alias('agora'))
        .filter(pl.col('antes') != pl.col('agora'))
        .with_columns(pl.when(pl.col('agora')).then(pl.lit('entrou')).otherwise(pl.lit('saiu')).alias('mudanca'))
        .select('mudanca', 'cnpj', 'razao_social', 'natureza_desc', pl.col('len').alias('registros'))
        .sort(['mudanca', 'registros'], descending=[False, True]))
    trans.write_csv(PASTA_SAIDA / 'transicao_regra_receita.csv')
    print(trans)
    sem_cnpj_ant = ant_reg - ant.filter(pl.col('cnpj').is_not_null()).height
    print(f'\nRegistros sem CNPJ que estavam na versão anterior e saíram: {sem_cnpj_ant:,}')

## 6. Memória de cálculo

In [9]:
t = tot.row(0, named=True)
print(f'Base harmonizada 2009-2023            : {t["total"]:>9,}')
print(f'  sem CNPJ do comprador (excluídos)   : {t["sem_cnpj"]:>9,}')
print(f'  natureza jurídica privada (excl.)   : {t["natureza_privada"]:>9,}')
print(f'  CNPJ não encontrado na Receita      : {t["nao_encontrado_receita"]:>9,}')
print(f'  BASE PÚBLICA                        : {t["publico"]:>9,}   (referência: 836.096 — após retirar as duplicatas exatas na rotina 03; antes 846.923)')
if ant_reg is not None:
    print(f'Versão anterior                       : {ant_reg:>9,}   (diferença: {t["publico"] - ant_reg:+,})')
assert t['total'] == t['publico'] + t['sem_cnpj'] + t['natureza_privada'] + t['nao_encontrado_receita']

Base harmonizada 2009-2023            :   872,804
  sem CNPJ do comprador (excluídos)   :       665
  natureza jurídica privada (excl.)   :    36,043
  CNPJ não encontrado na Receita      :         0
  BASE PÚBLICA                        :   836,096   (referência: 836.096 — após retirar as duplicatas exatas na rotina 03; antes 846.923)
